# LLM News Research Tool
## Notebook 4: Writing the summary

The summary step reads the top 5 articles of the default ranking (O1+S1) and writes, for an equity analyst:

- **key findings** about the company, each citing the articles it rests on;
- **industry context**, also cited;
- a short **brief** of each article;
- the **articles not used**, with the reason.

Two designs are compared. **Direct** sends one request that reads all 5 articles. **Brief-first** sends one request per article for its brief, then one request that combines the briefs.

**Evaluation boundary.** Development runs (three searches from October 2026) guide every design choice in this notebook. The fresh week of 24 to 30 October is kept apart: its labels and events are frozen before any model runs on it, and nothing is tuned on it.

Every table is computed from committed records or recomputed with the project's code. Publisher text stays in `data/fulltext/`, outside Git; where a table needs it and it is not on this computer, the committed record is shown instead.

In [1]:
import glob, json, os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
from src.config import DATA_DIR, EVAL_DIR
from src.query import load_aliases

pd.set_option("display.max_colwidth", 90)
ALIASES = load_aliases(DATA_DIR / "company_aliases.json")
IVECO_PAGE = DATA_DIR / "fulltext" / "9945d7a20638bf7b9e1143d9f9abbbafe758c0c1.json"
FULL_TEXT_HERE = (DATA_DIR / "fulltext").exists() and IVECO_PAGE.exists()

def record(name):
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## 1. What the model reads

Two different passages exist, and they serve different steps:

- The **relevance passage** (version 1) is what the relevance check and the ranking read in Stages 2 and 3. It is frozen with those evaluations and does not change in this stage.
- The **summary passage** is what the summary model reads. It starts from the relevance passage and changes in two steps below. Each step is accepted only after a check against the hand-made fact list.

### 1.1 Choosing the input: the fact check

For the 16 long development articles (longer than 3,500 characters), 77 important facts are written down before any input is compared; 36 of them are central. Three inputs are then marked blind (as X, Y and Z) for each fact: kept in full, partly, or not, and whether the meaning is distorted.

In [2]:
fc = record("fact_check_score.json")
rows = {}
for name, label in (("short", "NewsAPI short text"), ("passage", "relevance passage (version 1)"),
                    ("evidence", "summary evidence (an alternative selector)")):
    s = fc["scores"][name]
    rows[label] = {group: f"{s[group]['yes']}/{s[group]['facts']}" for group in ("central", "company", "context")} | \
                  {"distortions": s["central_distortions"] + s["other_distortions"]}
display(pd.DataFrame(rows).T)
assert fc["choice"] == "passage"
print("Chosen input:", fc["choice"], "(kept in full; the highest share of central facts, no distortion)")

,central,company,context,distortions
NewsAPI short text,4/36,4/64,2/13,0
relevance passage (version 1),31/36,50/64,11/13,0
summary evidence (an alternative selector),29/36,50/64,12/13,0


Chosen input: passage (kept in full; the highest share of central facts, no distortion)


### 1.2 Summary passage version 2: whole continued quotes

Version 2 adds, to the relevance passage, a whole paragraph that continues a quote after a company mention, within a hard cap of 4,200 characters. Three of the 16 passages change; only their facts are marked again.

In [3]:
p2 = record("fact_check_passage2_score.json")
display(pd.DataFrame({name: {g: f"{p2[name][g]['yes']}/{p2[name][g]['facts']}" for g in ("central", "company", "context")}
                      | {"distorted": p2[name]["all"]["distorted"]} for name in ("old", "new")}).T
        .rename(index={"old": "version 1 (relevance passage)", "new": "version 2"}))
assert p2["accepted"] and not p2["lost"] and not p2["distorted"]
print(f"Changed articles: {len(p2['changed_articles'])}; facts gained: {p2['gained']}; lost: {p2['lost']}")

,central,company,context,distorted
version 1 (relevance passage),31/36,50/64,11/13,0
version 2,32/36,51/64,11/13,0


Changed articles: 3; facts gained: [['Ad74eeaf8f2', '1']]; lost: []


### 1.3 Summary passage version 3: press-release boilerplate removed

A live search shows that a press release can fill most of the passage with text that is not news. Version 3 removes press-release boilerplate by the structure of the text (`src/boilerplate.py`), never by single words:

- a **boilerplate line** is a line inside a removed part: a distribution notice in capital letters at the very start (such as "NOT FOR RELEASE…"), or a **tail** that begins at a heading such as "Contacts", "About <Name>", "General restrictions" or "Comments" and holds only boilerplate up to the end of the text;
- a **news line** is any other line of the article.

The selector then runs on the cleaned text, and the room the boilerplate took goes to the article's own paragraphs. An article with no boilerplate gets exactly its version 2 passage. Version 3 is compared with version 2.

In [4]:
v3 = record("summary_passage_v3_check.json")
print(f"Fact-check passages changed (version 2 -> 3): {len(v3['fact_check']['changed'])} of {v3['fact_check']['articles']}")
print(f"Development top-10 passages changed: {len(v3['development_top10']['changed'])} of {v3['development_top10']['passages']}")
print(f"Saved pages with boilerplate: {v3['saved_pages']['with_boilerplate']} of {v3['saved_pages']['with_text']}")
display(pd.DataFrame(v3["saved_pages"]["pages"])[["site", "removed_chars", "v2_chars", "v3_chars", "news_lines_lost"]])
assert v3["accepted"]

Fact-check passages changed (version 2 -> 3): 0 of 16
Development top-10 passages changed: 0 of 30
Saved pages with boilerplate: 3 of 204


,site,removed_chars,v2_chars,v3_chars,news_lines_lost
0,www.thehindubusinessline.com,598,1090,472,0
1,financialpost.com,12047,3497,3480,0
2,www.prnewswire.com,1068,3498,3498,0


Because no fact-check passage changes, the fact-check results of version 2 hold for version 3 unchanged: no central fact is lost and no distortion is added. The fact list therefore cannot show the improvement itself. The evidence for the change is the pages that do have boilerplate: on each, every news line of version 2 is still in version 3.

The Iveco press release (Financial Post, 30 October 2026) is the page that prompts the change. Below, each line of its version 2 and version 3 passages is counted as news or boilerplate by the definitions above.

In [5]:
if FULL_TEXT_HERE:
    from src.boilerplate import clean, _paragraphs
    from src.relevance_check import build_summary_passage
    from src.summary_passage import build
    saved = json.loads(IVECO_PAGE.read_text(encoding="utf-8"))
    title = ("TATA MOTORS COMPLETES INITIAL SETTLEMENT OF RECOMMENDED TENDER OFFER FOR IVECO GROUP. "
             "IVECO GROUP HAS NOW BECOME PART OF TATA MOTORS, SUBSEQUENT CHANGES ARE ANNOUNCED")
    paragraphs = _paragraphs(saved["text"])
    removed = clean(saved["text"])["removed"]
    boilerplate = [p for r in removed for p in paragraphs[r["first"]:r["last"] + 1]]

    def split(passage):
        counts = {"news": 0, "boilerplate": 0}
        for line in passage.split("\n")[1:]:
            core = line.strip().strip("…").strip()
            if core and core != "[…]":
                counts["boilerplate" if any(core in b for b in boilerplate) else "news"] += len(core)
        return counts

    old = build_summary_passage(title, saved["text"], "Tata Motors", ALIASES)["text"]
    new = build(title, saved["text"], "Tata Motors", ALIASES)["text"]
    table = pd.DataFrame({"version 2": split(old), "version 3": split(new)}).T
    table["boilerplate share"] = (table["boilerplate"] / (table["news"] + table["boilerplate"])).round(2)
    display(table)
    assert split(new)["boilerplate"] == 0
    print("Removed parts:", [(r["kind"], r["heading"]) for r in removed])
else:
    page = next(p for p in v3["saved_pages"]["pages"] if p["site"] == "financialpost.com")
    print("The page is not on this computer; the committed check:", page)

,news,boilerplate,boilerplate share
version 2,1205,2086,0.63
version 3,3305,0,0.00


Removed parts: [('leading_notice', ''), ('tail', 'Contacts')]


For this passage, version 2 spends most of its room on boilerplate, while every line of version 3 comes from the news part of the release.

## 2. The automatic checks, and what they cannot see

Every answer goes through three checks before it is shown:

- **structure:** valid JSON, every required section, every article accounted for;
- **numbers:** every number in a point must appear in the articles that point cites (version 2 reads "14.40 euro" and "EUR 14.40" as €14.40, and keeps amounts, scales and currencies exact);
- **source naming:** a point names its sources only through its citations, never as "brief 2" or "the first article".

A point that fails is repaired once or twice; a point that still fails is removed with its reason.

In [6]:
from src.summarizer import source_references, unconfirmed_numbers

examples = [
    ("caught", "a number that is not in the cited article", "Sales rose 25% in October.", "Sales rose 23% in October."),
    ("caught", "another currency", "The fee is €20.", "The fee is 20 dollars."),
    ("caught", "another scale", "The deal is worth €3.91 billion.", "The deal is worth 3.91 million euro."),
    ("passes", "an equal amount written another way", "The offer is €14.40 a share.", "The offer rises to 14.40 euro per share."),
    ("limit", "the right numbers on the wrong measures", "Revenue fell 12% and sales rose 5%.", "Revenue rose 12% while sales fell 5%."),
    ("limit", "the right number for another company and month", "Tata Motors sales rose 23% in September.",
     "Mahindra sales rose 23% in October."),
    ("limit", "a claim with no number that the article does not make", "The deal faces no regulatory risk.",
     "The deal needs approval in Brazil."),
]
rows = []
for kind, what, claim, source in examples:
    flagged = unconfirmed_numbers(claim, source)
    rows.append({"expected": kind, "case": what, "point": claim, "cited article": source,
                 "numbers flagged": ", ".join(flagged) or "none"})
display(pd.DataFrame(rows))
assert [bool(r["numbers flagged"] != "none") for r in rows] == [True, True, True, False, False, False, False]
print("Source wording flagged:", source_references("Brief 2 says the offer rises, as the first article reports.", [1, 2, 3]))
print("Source wording in a cited point:", source_references("Tata Motors raises its offer.", [1, 2, 3]) or "none")

,expected,case,point,cited article,numbers flagged
0,caught,a number that is not in the cited article,Sales rose 25% in October.,Sales rose 23% in October.,25%
1,caught,another currency,The fee is €20.,The fee is 20 dollars.,€20
2,caught,another scale,The deal is worth €3.91 billion.,The deal is worth 3.91 million euro.,€3.91 billion
3,passes,an equal amount written another way,The offer is €14.40 a share.,The offer rises to 14.40 euro per share.,none
4,limit,the right numbers on the wrong measures,Revenue fell 12% and sales rose 5%.,Revenue rose 12% while sales fell 5%.,none
5,limit,the right number for another company and month,Tata Motors sales rose 23% in September.,Mahindra sales rose 23% in October.,none
6,limit,a claim with no number that the article does not make,The deal faces no regulatory risk.,The deal needs approval in Brazil.,none


Source wording flagged: ['Brief 2', 'first article']
Source wording in a cited point: none


The number check confirms that a number appears in a cited article. It does not establish that the number refers to the right measure, period, company or unit, and a claim with no number passes it whatever it says. The source-naming check only concerns how a point names its sources, not whether those sources support it. These limits are why the summaries are also judged by hand (section 5).

## 3. Errors found and fixed

Each problem below appears in a saved run. Each run is a new answer from the model (temperature 1), so two runs are never a controlled comparison; where a controlled check is possible, it is run on the saved answer itself.

### 3.1 Answers cut off at 3,072 tokens

Without an explicit output limit, some answers stop in the middle of their JSON with the finish reason `length`. The request carries no output limit, yet answers stop at 3,072 output tokens (hidden reasoning included). From configuration 3 on, every summary request sends an explicit limit of 4,096 tokens.

In [7]:
rows = []
for name in ("summary_v2_diagnosis_p3_run2.json", "summary_v2_diagnosis_c3_p3.json"):
    r = record(name)
    for number, call in enumerate(r["records"][0]["result"]["usage"]["per_call"], start=1):
        rows.append({"record": name, "output limit sent": r.get("output_limit_sent") or "none", "call": number,
                     "output tokens": call["output"], "of which reasoning": call["reasoning"],
                     "finish": call["finish_reason"]})
table = pd.DataFrame(rows)
display(table)
assert table.iloc[1]["output tokens"] == 3072 and table.iloc[1]["finish"] == "length"
assert table.iloc[3]["output tokens"] > 3072 and table.iloc[3]["finish"] == "stop"

,record,output limit sent,call,output tokens,of which reasoning,finish
0,summary_v2_diagnosis_p3_run2.json,none,1,2121,1097,stop
1,summary_v2_diagnosis_p3_run2.json,none,2,3072,2091,length
2,summary_v2_diagnosis_c3_p3.json,4096,1,1728,573,stop
3,summary_v2_diagnosis_c3_p3.json,4096,2,3154,2154,stop


With the limit sent, the correction answer reaches 3,154 tokens and finishes. Every later recorded answer finishes with `stop` too (sections 3.3 and 3.4). An answer that still reaches the limit is caught by the structure check.

The top-level `version` field of `summary_v2_diagnosis_c3_p3.json` reads "4"; the record's `result.correction` ("full") and its file name identify configuration 3.

### 3.2 Full correction and targeted repair

With **full correction**, any failing point sends the whole summary back to the model, so every point is written again, including the points that already pass. **Targeted repair** sends only each failing point back, with its articles, at most twice; the other points stay exactly as they are. Broken JSON or a missing section still gets the full correction.

In [8]:
rows = []
for name in ("summary_v2_diagnosis_c3_p3.json", "summary_v2_diagnosis_c5_p3.json"):
    result = record(name)["records"][0]["result"]
    log = result["attempt_log"]
    rows.append({"record": name, "correction": result["correction"],
                 "points failing, first answer": len(log[0]["numbers"]) + len(log[0]["sources"]),
                 "full rewrites": len(log) - 1, "repair requests": sum(r.get("requests", r.get("tries", 0)) for r in result["repairs"]),
                 "points removed": len(result["removed"]), "status": result["status"], "failures": result.get("failures") or "none"})
display(pd.DataFrame(rows))

,record,correction,"points failing, first answer",full rewrites,repair requests,points removed,status,failures
0,summary_v2_diagnosis_c3_p3.json,full,5,1,1,2,partial,[daily_limit]
1,summary_v2_diagnosis_c5_p3.json,targeted,1,0,1,0,ok,none


In the full-correction run, the rewrite still fails on two points, and its repair is then stopped by the daily token limit. In the targeted run, one point fails and one repair fixes it. Section 3.3 shows what that repair changes.

### 3.3 Correct euro amounts flagged as wrong (number check version 2)

In the targeted runs with number check version 1, amounts such as "€14.40" are flagged although the cited article states them, because the article writes "14.40 euro". When the daily limit stops the repair, those points are removed, and correct facts are lost.

In [9]:
rows = []
for name in ("summary_v2_diagnosis_c4_p3.json", "summary_v2_diagnosis_c5_p3.json"):
    r = record(name)
    result = r["records"][0]["result"]
    rows.append({"record": name, "number check": r.get("number_check"),
                 "points flagged": len(result["attempt_log"][0]["numbers"]), "points removed": len(result["removed"]),
                 "status": result["status"], "failures": result.get("failures") or "none"})
display(pd.DataFrame(rows))

,record,number check,points flagged,points removed,status,failures
0,summary_v2_diagnosis_c4_p3.json,1,3,3,partial,[daily_limit]
1,summary_v2_diagnosis_c5_p3.json,2,1,0,ok,none


In [10]:
c4 = record("summary_v2_diagnosis_c4_p3.json")["records"][0]["result"]
if FULL_TEXT_HERE:
    from src.summarizer import _article_text
    from src.summary_v2 import DIAGNOSIS_RUN, DEV_K, _dev_candidates, summary_articles
    runs, dev_aliases = _dev_candidates()
    articles = {a["id"]: a for a in summary_articles(runs[DIAGNOSIS_RUN], DEV_K, dev_aliases)}
    brief = next(r for r in c4["removed"] if r["section"] == "article_briefs")
    print(f"Removed brief of article {brief['id']}, under number check version 2:",
          unconfirmed_numbers(brief["text"], _article_text(articles[brief["id"]])) or "every number confirmed")
    twenty = next(r for r in c4["removed"] if "20%" in r["text"])
    found = {n: unconfirmed_numbers(twenty["text"], _article_text(a)) for n, a in articles.items()}
    print("The '20%' point, checked against each article alone:", found)
    assert unconfirmed_numbers(brief["text"], _article_text(articles[brief["id"]])) == []
    assert all("20%" in v for n, v in found.items() if n != 2)
else:
    print("The articles are not on this computer; the removed points:", [(r["section"], r["reason"]) for r in c4["removed"]])

Removed brief of article 1, under number check version 2: every number confirmed
The '20%' point, checked against each article alone: {1: ['20%', '28.4%'], 2: ['28.4%'], 3: ['20%', '28.4%'], 4: ['20%', '28.4%'], 5: ['20%']}


On the saved answer itself, the removed brief passes number check version 2 against its own article, while the "approximately 20%" point still fails against every article except article 2, which it does not cite (its articles say 20.4%). Version 2 removes the false alarm and keeps the real error.

### 3.4 Requests refused by the project's own pacer

The pacer keeps requests under the free plan's limits (8,000 tokens a minute for this model). Version 1 counts every request as its input plus the whole 4,096-token output limit. In the first comparison at configuration 7, two of the three direct requests are refused **before sending**:

In [11]:
c7 = record("summary_v2_dev_k5_c7_p3.json")
rows = [{"run": e["run"], "design": e["design"], "status": e["result"]["status"],
         "requests": [(q["estimated"], q["outcome"]) for q in e["result"]["usage"].get("requests", [])],
         "calls": e["result"]["usage"].get("calls")} for e in c7["records"]]
display(pd.DataFrame(rows))
print("Pacer version of this record:", c7["pacer_version"])
calls = record("summary_v2_diagnosis_c5_p3.json")["records"][0]["result"]["usage"]["per_call"]
print(f"Provider header after the first call of the configuration 5 diagnosis: {calls[1]['provider_remaining']} of 8,000 "
      f"tokens left (first call: {calls[0]['input']} input, {calls[0]['output']} output tokens)")

,run,design,status,requests,calls
0,2026-10-14,direct,ok,"[(9276, answered), (5062, answered)]",2
1,2026-10-14,brief_first,ok,"[(4850, answered), (5120, answered), (4809, answered), (4821, answered), (5248, answer...",6
2,2026-10-23,direct,unavailable,"[(8060, preflight_too_large)]",0
3,2026-10-23,brief_first,ok,"[(5188, answered), (5202, answered), (4714, answered), (5005, answered), (5125, answer...",7
4,2026-10-24,direct,unavailable,"[(9136, preflight_too_large)]",0
5,2026-10-24,brief_first,ok,"[(5259, answered), (6198, answered), (6121, answered), (6181, answered), (6219, answer...",6


Pacer version of this record: 1
Provider header after the first call of the configuration 5 diagnosis: 4521 of 8,000 tokens left (first call: 3186 input, 2242 output tokens)


The refused requests never reach Groq, so they say nothing about Groq's own limits. The header after the first call of the configuration 5 diagnosis is consistent with Groq counting less than the input plus the whole output limit, but a header is an organisation-wide, rolling figure, not an exact receipt for one request.

Pacer version 2 therefore removes only the refusal that the project makes itself. The rate check uses the input estimate plus a fixed margin of 600 tokens; the output limit is still sent with every request and is checked against the model's context window; before any real token count is known, a large request goes alone and Groq decides. Real use is settled after every answer, as before. Whether Groq accepts larger requests is shown only by real requests (the comparison in section 5 and the k = 10 test in section 8).

Pacer version 3 changes one case: once the token rate is learned, a request whose input fits the minute limit and is over only because of the 600-token margin waits until it can be sent alone, and Groq decides. The second k = 10 record and the fresh-week run use version 3, and the freeze fixes it.

## 4. Configuration history

Each configuration fixes the summary settings of a run. A configuration changes when any of these changes; the record files keep the configuration they are made with. The settings come from `src/summary_v2.py`, and the second table reads every saved record.

In [12]:
from src.summary_v2 import config_version
history = [
    ("2", "prompt 2, then 3", "none", "full", "1", "2", "1"),
    (config_version("full", "1", "2"), "3", "4,096", "full", "1", "2", "1"),
    (config_version("targeted", "1", "2"), "3", "4,096", "targeted", "1", "2", "1"),
    (config_version("targeted", "2", "2"), "3", "4,096", "targeted", "2", "2", "1"),
    (config_version("full", "2", "2"), "3", "4,096", "full", "2", "2", "-"),
    (config_version("targeted", "2", "3"), "3, then 4 and 5", "4,096", "targeted", "2", "3", "1, then 2, then 3"),
    (config_version("full", "2", "3"), "-", "4,096", "full", "2", "3", "-"),
]
table = pd.DataFrame(history, columns=["configuration", "prompt", "output limit", "correction", "number check",
                                       "summary passage", "pacer"]).set_index("configuration")
display(table)

,prompt,output limit,correction,number check,summary passage,pacer
configuration,,,,,,
2,"prompt 2, then 3",none,full,1,2,1
3,3,"4,096",full,1,2,1
4,3,"4,096",targeted,1,2,1
5,3,"4,096",targeted,2,2,1
6,3,"4,096",full,2,2,-
7,"3, then 4 and 5","4,096",targeted,2,3,"1, then 2, then 3"
8,-,"4,096",full,2,3,-


In [13]:
def configuration(name, r):
    if "_c" in name:
        return name.split("_c")[1].split("_")[0]
    return str(r.get("version"))

rows = []
for path in sorted(glob.glob(str(EVAL_DIR / "summary_v2_*.json"))):
    name = Path(path).name
    r = json.loads(Path(path).read_text(encoding="utf-8"))
    if "records" not in r:
        continue
    rows.append({"record": name, "configuration": configuration(name, r),
                 "version field": r.get("version"), "pacer": r.get("pacer_version", "1"),
                 "runs": ", ".join(sorted({e["run"] for e in r["records"]})),
                 "results": ", ".join(f"{e['design']} {e['result']['status']}" for e in r["records"])})
display(pd.DataFrame(rows).set_index("record"))

,configuration,version field,pacer,runs,results
record,,,,,
summary_v2_dev_k10_c7_p4.json,7,7,2,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, direct unavailable, direct ok"
summary_v2_dev_k10_c7_p4_run2.json,7,7,3,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, direct unavailable, direct unavailable"
summary_v2_dev_k5_c7_p3.json,7,7,1,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, brief_first ok, direct unavailable, brief_first ok, direct unavailable, bri..."
summary_v2_dev_k5_c7_p3_run2.json,7,7,2,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, brief_first ok, direct ok, brief_first ok, direct ok, brief_first ok"
summary_v2_dev_k5_c7_p4.json,7,7,2,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, direct ok, direct ok"
summary_v2_dev_k5_c7_p5.json,7,7,2,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, direct ok, direct ok"
summary_v2_dev_k5_c7_p6.json,7,7,3,"2026-10-14, 2026-10-23, 2026-10-24","direct ok, direct ok, direct unavailable"
summary_v2_dev_k5_p2.json,2,2,1,"2026-10-14, 2026-10-23, 2026-10-24","direct unavailable, brief_first ok, direct unavailable, brief_first ok, direct unavail..."
summary_v2_dev_k5_p3.json,2,2,1,"2026-10-14, 2026-10-23, 2026-10-24","direct unavailable, brief_first ok, direct ok, brief_first unavailable, direct unavail..."


Configurations 6 and 8 (full correction with number check 2, on summary passage 2 and 3) are defined but not run. The records of configuration 2 come from before the explicit output limit and are kept as the evidence for section 3.1. A record made with pacer version 1 has no `pacer_version` field unless it is marked afterwards; the first configuration 7 comparison is marked. The fresh-week record has no version field; it names its configuration in its file name and carries the freeze fingerprint.

## 5. Design comparison and final choice

### 5.1 What is compared

Both designs run on the three development runs (14, 23 and 24 October) with configuration 7 and the same five articles per run (`summary_v2_dev_k5_c7_p3_run2.json`). The direct design writes the whole summary in one call. The brief-first design writes one brief per article from that article alone, then combines the briefs into the summary.

Every item of every summary (company points, industry points and article briefs) is judged against its cited articles, following `SUMMARY_JUDGING_GUIDE.md`. The sheet is shuffled and never shows which design wrote an item; repaired items are marked only in the hidden key. Event coverage is judged per summary: for each relevant event among the summary's articles, the question is whether the summary states it. The judgements are locked before the key is opened.

### 5.2 Results

In [14]:
score = record("summary_v2_judging_score.json")
rows = {}
for design, d in score["designs"].items():
    i, e = d["items"], d["events"]
    rows[design] = {"summaries produced": f"{d['produced']['count']}/{d['produced']['total']}",
                    "items judged": i["total"],
                    "supported": f"{i['supported']} ({i['supported_share']:.1%})",
                    "partly supported": i["partly_supported"], "not supported": i["not_supported"],
                    "contradicted": i["contradicted"],
                    "events covered": f"{e['covered']}/{e['total']} ({e['coverage']:.1%})",
                    "calls per summary": d["calls"]}
display(pd.DataFrame(rows)[["brief_first", "direct"]])
print("Choice:", score["choice"])

,brief_first,direct
summaries produced,3/3,3/3
items judged,30,45
supported,18 (60.0%),31 (68.9%)
partly supported,10,10
not supported,2,4
contradicted,0,0
events covered,10/11 (90.9%),11/11 (100.0%)
calls per summary,7.0,2.33


Choice: direct


Both designs meet every acceptance rule: at least 90% of summaries produced, event coverage of at least 80%, at most 5% of items contradicted, and no central contradiction. The choice rule takes the higher supported share, then the higher event coverage, then fewer calls. Direct is ahead on all three, so **direct is the chosen design.**

In [15]:
import csv
def sheet(name):
    with open(EVAL_DIR / name, encoding="utf-8-sig", newline="") as handle:
        return {row["row"]: row for row in csv.DictReader(handle)}

items, key = sheet("summary_v2_judging.csv"), record("summary_v2_judging_key.json")
per_run = {}
for number, entry in key["items"].items():
    cell = per_run.setdefault(entry["run"], {}).setdefault(entry["design"], [0, 0])
    cell[0] += items[number]["judgement"] == "supported"
    cell[1] += 1
display(pd.DataFrame({run: {design: f"{s}/{t} ({s / t:.1%})" for design, (s, t) in designs.items()}
                      for run, designs in sorted(per_run.items())}).T[["brief_first", "direct"]])

,brief_first,direct
2026-10-14,8/10 (80.0%),10/11 (90.9%)
2026-10-23,7/11 (63.6%),10/17 (58.8%)
2026-10-24,3/9 (33.3%),11/17 (64.7%)


Direct has the higher supported share on two of the three runs. On 23 October, brief-first is higher.

### 5.3 What the repairs show

Six items come from a repair. The four number repairs all end as supported. One of them catches a figure in Indian digit grouping: the article states 1,35,114 units (1.35 lakh), and the first answer turns it into "about 1.35 million units", ten times too large. The number check finds that 1.35 million is not in the article, and the repair restores 1,35,114.

The two source repairs remove references such as "brief 1" from the text. Both points still claim a "discrepancy" between €3.8 billion and €3.91 billion, which are two stages of the deal, not conflicting reports, so both stay not supported. A source repair fixes the form of a point, not its claim. Both points come from the brief-first design: combining short briefs without the full articles makes normal changes in a deal look like conflicts.

### 5.4 Where the direct design goes wrong

In [16]:
by_section = {}
for number, entry in key["items"].items():
    if entry["design"] == "direct":
        cell = by_section.setdefault(entry["section"], {"supported": 0, "partly_supported": 0, "not_supported": 0})
        cell[items[number]["judgement"]] += 1
display(pd.DataFrame(by_section).T.loc[["key_points", "article_briefs", "industry_context"]])

,supported,partly_supported,not_supported
key_points,18,5,0
article_briefs,11,4,0
industry_context,2,1,4


All four not-supported direct items are industry-context points, and each one states a Tata Motors fact as a trend for the sector: Tata's own commercial-vehicle volumes as growth of the whole Indian segment, Tata's margin pressure as pressure across the sector, one deal's acceptance threshold as a wider trend in mergers, and the combined size of one deal as a change to the European market. Key points and briefs have no not-supported item (29 of 38 supported). Section 6 tests two fixes for this weakness.

## 6. Industry-context fix

### 6.1 The weakness

In the design comparison, every unsupported direct point is an industry point that states a Tata Motors fact as a sector trend (section 5.4). Key points and briefs have no unsupported item, so the fix targets the industry rule only.

### 6.2 Prompt version 4

Prompt version 4 adds one rule to both designs:

> An "industry_context" point needs an article that itself reports on the wider industry, the market or other companies. A fact about the company alone (its own sales, margins, deals or plans) is never industry context and is never stated as a trend for the sector. A reference to another company only as a party to a deal, partnership or transaction involving the company does not count as wider-industry evidence. When no article reports on the wider industry, "industry_context" is an empty list.

The direct design runs again on the same three development runs, once each. Every item is judged blind with the same guide. For each run, a separate question is judged from the five articles alone, before the summary is read: does any article report a fact about the sector, the market, or another company in its own right (`SUMMARY_JUDGING_GUIDE_PART4.md`)? The pass rule needs no unsupported industry point, an industry section that matches that answer, key points and briefs no more than 5 percentage points below the comparison result, and event coverage of at least 80%.

In [17]:
def version_row(result):
    t, kb = result["total"], result["key_points_and_briefs"]
    return {"unsupported industry points": t["industry"]["not_supported"],
            "industry points supported / partly": f"{t['industry']['supported']} / {t['industry']['partly_supported']}",
            "key points and briefs supported": f"{kb['supported']} of {kb['total']} ({kb['share']:.1%})",
            "event coverage": f"{t['events']['covered']} of {t['events']['total']}"}

direct = score["designs"]["direct"]
comparison = {"unsupported industry points": 4, "industry points supported / partly": "2 / 1",
              "key points and briefs supported": "29 of 38 (76.3%)",
              "event coverage": f"{direct['events']['covered']} of {direct['events']['total']}"}
p4 = record("summary_v2_p4_judging_score.json")
display(pd.DataFrame({"comparison (prompt 3)": comparison, "prompt 4": version_row(p4)}))
print({check: ("pass" if ok else "FAIL") for check, ok in p4["checks"].items()})

,comparison (prompt 3),prompt 4
unsupported industry points,4,0
industry points supported / partly,2 / 1,1 / 0
key points and briefs supported,29 of 38 (76.3%),32 of 40 (80.0%)
event coverage,11 of 11,11 of 11


{'no_unsupported_industry_point': 'pass', 'empty_section_only_without_evidence': 'FAIL', 'industry_points_only_with_evidence': 'pass', 'a_supported_industry_point_or_all_empty_without_evidence': 'pass', 'no_unsupported_key_point_or_brief': 'pass', 'key_points_and_briefs_within_tolerance': 'pass', 'event_coverage': 'pass', 'every_development_run_has_one_summary': 'pass'}


The rule removes every unsupported industry point, and key points and briefs improve. One check fails. On 24 October, one article reports a quarterly result of another automotive company (JK Tyre, a tyre maker, in a results roundup), but the industry section is empty. The rule says what is not industry context; it never tells the model to include a real sector fact, so the model leaves the section empty.

The number check finds the same digit-grouping error as in section 5.3: "~1.36 million units" for an article's 1,35,114 to 1,37,339 units. The repair restores the figures as the article writes them.

### 6.3 Prompt version 5

Prompt version 5 keeps the rule and adds one instruction:

> When an article reports a relevant fact about the company's sector or market, or about another company operating in that same sector, include that fact as an "industry_context" point and cite the article. Examples include a rival's sales, a sector-wide figure, or a supplier's results. Do not include companies from other sectors merely because an article lists them next to the company, for example in a "stocks in news" list or a results roundup.

The test repeats with the same runs and rules (`SUMMARY_JUDGING_GUIDE_PART5.md`). Each run keeps its industry-evidence answer from prompt version 4, because the articles are the same.

In [18]:
p5 = record("summary_v2_p5_judging_score.json")
display(pd.DataFrame({"prompt 4": version_row(p4), "prompt 5": version_row(p5)}))
print({check: ("pass" if ok else "FAIL") for check, ok in p5["checks"].items()})

,prompt 4,prompt 5
unsupported industry points,0,0
industry points supported / partly,1 / 0,2 / 3
key points and briefs supported,32 of 40 (80.0%),29 of 36 (80.6%)
event coverage,11 of 11,11 of 11


{'no_unsupported_industry_point': 'pass', 'empty_section_only_without_evidence': 'pass', 'industry_points_only_with_evidence': 'FAIL', 'a_supported_industry_point_or_all_empty_without_evidence': 'pass', 'no_unsupported_key_point_or_brief': 'pass', 'key_points_and_briefs_within_tolerance': 'pass', 'event_coverage': 'pass', 'every_development_run_has_one_summary': 'pass'}


The opposite check now fails. On 14 October, no article reports on the wider industry, but the summary gives two industry points: one about Stellantis's brand portfolio (Stellantis appears only as Tata's partner) and one drawn from an analyst's remark about Iveco's position if the deal fails. On 24 October, the JK Tyre result is still missing, and two weak points appear instead.

### 6.4 Selection

Neither version passes every check. Prompt version 4 is selected because it has fewer industry points that are not fully supported, meaning judged partly supported or not supported (0 of 1, against 3 of 5). This selection rule is stated after both results, not set in advance.

The development results show a trade-off in industry-context reporting. Prompt version 4 avoids unsupported industry claims but misses a relevant fact about another company in the same sector (one case in the three development runs). Prompt version 5 includes more industry points but also adds weak points that do not fully meet the judging standard. The choice comes from these three runs and does not prove that prompt version 4 is better on unseen news; section 9 tests it on a fresh week. Further tuning on the same three runs stops here, so the prompt is not fitted to them.

## 7. Consistency of the judgements

After the comparison is locked, 15 items (20% of 75) are judged a second time by the same judge: chosen at random with a fixed seed, in a new order, with new row numbers and without the first judgements. The first judgements stay the ones used for every result.

In [19]:
import math
rejudge = record("summary_v2_rejudge_score.json")
levels = ["supported", "partly_supported", "not_supported"]
display(pd.DataFrame(rejudge["table"]).T.loc[levels, levels].rename_axis("first \\ second"))

n = rejudge["items"]
matches = sum(rejudge["table"][level][level] for level in levels)
p, z = matches / n, 1.96
centre = (p + z * z / (2 * n)) / (1 + z * z / n)
half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
print(f"Matching judgements: {matches} of {n} ({p:.1%}); approximate 95% interval {centre - half:.0%} to {centre + half:.0%}")
print(f"Matching contradiction decisions: {rejudge['contradicted_agreement']:.0%}")

,supported,partly_supported,not_supported
first \ second,,,
supported,10,1,0
partly_supported,1,2,0
not_supported,0,0,1


Matching judgements: 13 of 15 (86.7%); approximate 95% interval 62% to 96%
Matching contradiction decisions: 100%


The two judgements match on 13 of 15 items (86.7%; approximate 95% interval 62% to 96%), and all 15 contradiction decisions match. This is the consistency of one judge across two passes, not agreement between independent judges. With 15 items the interval is wide.

Both differences sit on the boundary between supported and partly supported. One is whether "sales" is an acceptable word for dispatches; the other is whether the timing of dividends that are already paid changes the support level. No item moves to or from not supported, and no contradiction decision changes. Both items belong to the direct design and move in opposite directions, so direct stays at 31 of 45 supported and the design choice does not change.

## 8. Articles per summary (k = 10)

More articles could give the summary more distinct events. The condition for testing k = 10 is fixed first: k = 10 is selectable only if every k = 10 request is answered and leaves at least 10% headroom under the free plan's limit of 8,000 tokens a minute, so at most 7,200 tokens per request.

In [20]:
k10 = [record("summary_v2_dev_k10_c7_p4.json"), record("summary_v2_dev_k10_c7_p4_run2.json")]
dates = {}
for run_record in k10:
    for entry in run_record["records"]:
        usage = entry["result"]["usage"]
        first = usage["requests"][0] if usage["requests"] else {}
        row = dates.setdefault(entry["run"], {"characters in the request": first.get("prompt_chars"),
                                              "input tokens counted by Groq": "-",
                                              "outcomes": []})
        if usage["per_call"]:
            row["input tokens counted by Groq"] = usage["per_call"][0]["input"]
        row["outcomes"].append("answered" if first.get("outcome") == "answered" else "not sent")
table = pd.DataFrame(dates).T
table["outcomes"] = table["outcomes"].apply(", ".join)
display(table)

,characters in the request,input tokens counted by Groq,outcomes
2026-10-14,30173,6083,"answered, answered"
2026-10-23,33162,-,"not sent, not sent"
2026-10-24,29803,6900,"answered, not sent"


In Groq's own refusal messages, the requested tokens are 276 to 551 above the input count; this is an observed range, not a formula. For 24 October, the largest observed difference gives a conservative estimate of 7,451 tokens, above 7,200; the smallest gives 7,176. The 23 October request has a longer input still. Groq does not refuse it; it is not sent because the project's estimate exceeds the limit.

k = 10 does not qualify under the free-plan headroom rule, so k = 5 stays. This is a limit of the free plan's 8,000 tokens a minute, not a measured loss of quality: the quality of k = 10 summaries is not judged.

## 9. Freeze and fresh-week test

### 9.1 The freeze

The selected configuration is fixed in `summary_freeze.json` and tagged `freeze-summary` in Git before any fresh-week run. It records the design, the validated k, every setting the summary depends on, the fingerprints of the summary code and the requirements, the package versions, the 13 development evidence files and the written decision (`summary_freeze_decision.md`).

In [21]:
freeze = record("summary_freeze.json")
print("Design:", freeze["design"], "| validated k:", freeze["ks"], "| tag:", freeze["tag"])
settings = dict(freeze["settings"])
settings.update({f"generation: {name}": value for name, value in settings.pop("generation").items()})
display(pd.Series(settings, name="value").to_frame())
print("Evidence files:", len(freeze["evidence"]))

Design: direct | validated k: [5] | tag: freeze-summary


,value
provider,groq
model,openai/gpt-oss-120b
prompt_version,4
configuration_version,7
correction,targeted
repair_tries,2
max_attempts,2
number_check,2
source_check,1
passage_version,3


Evidence files: 13


Any later use of the summary checks the freeze first and refuses when a setting, a code file or k differs. A request for k = 10 is refused because the freeze validates k = 5 only.

### 9.2 The fresh-week test

The week of 24 to 30 October is not used to select or tune the summary prompt, configuration or passage rules. Stage 3 evaluates the ranking method on this same week separately; its labels and event sheet come from Stage 3 and are not changed here.

The summary runs once. Before the model is called, the run checks that it has not been attempted before, that the freeze accepts the current code and settings, and that no protected file has uncommitted changes. It then writes an attempt marker, so the run can never be sent a second time, and saves the exact passages the model receives outside Git, with their fingerprints in the record. The default ranking selects the top 5 of 36 candidates. The run uses one model call and 5,734 tokens, with no failure, no repair and no removed point.

The summary is judged blind with the rules of `SUMMARY_JUDGING_GUIDE_PART6.md`: industry evidence first, from the five articles alone; then every item; then event coverage. The judgements are locked before the score is made, and the lock covers the sheets, the key, the guides, the freeze record, the run record, the saved passages, the fresh labels and events, and the judging code.

### 9.3 Results

In [22]:
fresh_run = record("summary_v2_fresh_k5_c7_p4.json")
result = fresh_run["records"][0]["result"]
usage = result["usage"]
print(f"Candidates: {len(fresh_run['candidates'])} | selected: {len(fresh_run['selected'])} | calls: {usage['calls']} | "
      f"tokens: {usage['input_tokens'] + usage['output_tokens']:,} | repairs: {len(result['repairs'])} | "
      f"removed points: {len(result['removed'])} | failures: {len(result['failures'])}")

fresh = record("summary_v2_fresh_judging_score.json")
display(pd.Series({check: "pass" if ok else "FAIL" for check, ok in fresh["checks"].items()}, name="result").to_frame())

total = fresh["total"]
industry, events, kb = total["industry"], total["events"], fresh["key_points_and_briefs"]
print(f"Event coverage: {events['covered']} of {events['total']} ({events['coverage']:.0%})")
print(f"Contradicted items: {fresh['contradicted']['count']} of {fresh['contradicted']['total']}")
print(f"Industry points: {industry['total']}, supported {industry['supported']}; "
      f"industry evidence: {fresh['per_run'][fresh_run['run']]['industry_evidence']}")
print(f"Key points and briefs supported: {kb['supported']} of {kb['total']} ({kb['share']:.1%}); development: {kb['development']}")

Candidates: 36 | selected: 5 | calls: 1 | tokens: 5,734 | repairs: 0 | removed points: 0 | failures: 0


,result
summary_produced,pass
event_coverage_at_least_80_percent,FAIL
contradicted_at_most_5_percent,pass
no_central_contradiction,pass
no_unsupported_industry_point,pass
industry_section_matches_evidence,pass


Event coverage: 3 of 4 (75%)
Contradicted items: 0 of 14
Industry points: 2, supported 2; industry evidence: yes
Key points and briefs supported: 9 of 12 (75.0%); development: 32/40 (80.0%)


Key points and briefs (reported, no pass level): 9 of 12 supported (75.0%), against 32 of 40 (80.0%) on the development runs. With 12 items, one item moves this share by about 8 points, so the two figures are not a reliable comparison.

### 9.4 Reading the result

The frozen design fails one check on the fresh week: event coverage. The missed event is the 26 October deadline outlook, an article written before the result. The summary reports the outcome of the same deadline instead (87.81% of shares accepted, the 80% threshold cleared, settlement and control), and it uses the outlook article for other points (the final price of €14.40 and an activist investor's valuation), but no point states the outlook itself. With four events, a single missed event gives 75%.

The industry-context rule works on the fresh week. The articles hold two real sector facts, about European truckmakers' costs and competition and about India's freight demand, and the summary reports both, each supported, with no company fact stated as a sector trend. No item is contradicted; the three partly supported items each overstate one detail: a first settlement called final, the article's own statement attributed to observers, and a collaboration called a signed agreement.

The frozen design stays as it is, whatever the result. A change, for example keeping the main point of an outlook article when a later article reports the outcome, belongs to a new development cycle and needs a new fresh week to test it.

## 10. A second development cycle: an expectation and its outcome

### 10.1 The problem and the change

The fresh week (section 9.4) misses one event: an outlook article written before a deadline. The summary reports the outcome of the same deadline but never states the outlook itself. Two development days hold the same kind of pair: on 24 October, a results preview and the actual results; on 23 October, a stage of shareholder acceptance and the final approval.

Prompt version 6 is prompt version 4 with two changes:

- **a new rule:** when one article reports what is expected or an earlier stage of a development, and another reports what then happens in the same development, each gets its own point. The expectation is stated as an expectation, with who expects it, and the outcome as the outcome. Articles about different developments are not linked by this rule;
- **one sentence added to the industry rule:** an industry point states only what the article says about the sector, the market or other companies, and never extends the company's own results to them. This answers an overreach seen in the test on a second company, Reliance Industries.

The other rules are unchanged. The frozen files are not changed: the test applies prompt 6 to its own requests only.

### 10.2 One run on the development days

The test, its judging and its choice rule are fixed and tagged before any output exists (`summary_v2_p6_plan.json`, tag `freeze-prompt6-plan`). Prompt 6 runs once on the same five articles per day as the locked prompt 4 run, with the same settings.

**The choice rule.** For each pair, a summary gets one point if it states the expectation as an expectation, with who expects it, and one point if it keeps the expectation and the outcome in separate points, with the outcome stated as the outcome. That gives 0 to 2 per pair, judged blind with both prompts' summaries mixed together. Prompt 6 replaces prompt 4 only if its pair score is not lower on either pair, its total is higher, and every guardrail holds:

| Guardrail | Condition for prompt 6 |
|---|---|
| Summaries | every day gives a usable summary |
| Industry points | none is unsupported; none appears without industry evidence; days with an empty industry section despite evidence are no more than prompt 4's |
| Overreach | industry points that extend the company's own results to the sector are no more than prompt 4's |
| Key points and briefs | none is unsupported; at least 75% are supported |
| Contradictions | no contradicted item |
| Events | event coverage on each day is not lower than prompt 4's |

A tie keeps prompt 4. **If a day gives no usable summary, prompt 4 stays, nothing is judged, and the run is not repeated.**


In [23]:
p6_run = record("summary_v2_dev_k5_c7_p6.json")
rows = []
for day in p6_run["records"]:
    result = day["result"]
    for attempt in result.get("attempt_log", []):
        problems = [f"structure: {p}" for p in attempt["structural"]]
        problems += [f"number check: {p['section'].replace('_', ' ')} {p['index'] + 1}" for p in attempt["numbers"]]
        problems += [f"source naming: {p['section'].replace('_', ' ')} {p['index'] + 1}" for p in attempt["sources"]]
        rows.append({"day": day["run"], "status": result["status"], "attempt": attempt["attempt"],
                     "valid JSON": attempt["valid_json"], "check problems": ", ".join(problems) or "none",
                     "repairs": len(result.get("repairs") or []), "calls for the day": result["usage"]["calls"]})
display(pd.DataFrame(rows).set_index(["day", "attempt"]))

decision = record("summary_v2_p6_judging_score.json")["choice"]
print("Saved decision: prompt", decision["prompt"], "|", decision["reason"], "|", "; ".join(decision["missing"]))
print("Judging sheets made:", any((EVAL_DIR / name).exists() for name in
                                  ("summary_v2_p6_pairs.csv", "summary_v2_p6_judging.csv", "summary_v2_p6_events.csv")))
print("Rule for a day without a summary:", record("summary_v2_p6_plan.json")["rule"]["failed_day"])


status  valid JSON                  check problems  \
day        attempt                                                            
2026-10-14 1                 ok        True                            none   
2026-10-23 1                 ok        True  number check: article briefs 5   
2026-10-24 1        unavailable        True  structure: unaccounted_article   
           2        unavailable        True  structure: unaccounted_article   

                    repairs  calls for the day  
day        attempt                              
2026-10-14 1              0                  1  
2026-10-23 1              1                  2  
2026-10-24 1              0                  2  
           2              0                  2

Saved decision: prompt 4 | failed day | prompt 6, 2026-10-24: no usable summary
Judging sheets made: False
Rule for a day without a summary: if any expected day has no usable summary (status ok or partial, with a summary) in either version, prompt 4 remains selected, no sheet is judged, and the run is not repeated


On 14 and 23 October, prompt 6 gives a usable summary on its first attempt; on 23 October, one figure in an article brief is flagged by the number check and fixed by the targeted repair. On 24 October it gives none. Its first answer leaves one article neither cited nor listed as not used, so the structure check fails. The corrected answer makes the same mistake, so the day ends without a summary, as the pipeline is built to do. **By the rule, prompt 4 stays, and nothing is judged.**

One output cannot show whether prompt 6 causes this failure. The same failure appears without prompt 6: in the first prompt 3 development run, the direct design fails on 23 and 24 October, and on neither in the repeat. The answers of failed attempts are not saved, so the cause stays open. The next test (10.3) runs each prompt several times and saves every answer.


### 10.3 Three matched runs per prompt

The test, its judging and its choice rule are fixed and tagged before any run (`summary_v2_r3_plan.json`, tag `freeze-round3-plan`). Each prompt runs three times on the same five articles per day, in three matched pairs; a fixed seed sets which prompt runs first in each pair. Every answer is saved. A token guard stops a run before it can pass 45,000 tokens, and a pair starts only when both runs fit inside Groq's daily limit of 200,000 tokens.

All 18 run-days give a usable summary. The summaries are judged blind, with both prompts mixed under letters. Sheet 1 holds the expectation-and-outcome pairs and every industry point, checked for overreach. Sheet 2 holds every key point and brief of one run per prompt and day, and the event coverage of every run. Ten items from earlier locked judging are mixed in unmarked to check consistency; they never count in the choice.

In [24]:
r3 = record("summary_v2_r3_judging_score.json")
runs, full = r3["all_runs"], r3["full_item"]


def covered(m, day=None):
    days = [m["events"][day]] if day else m["events"].values()
    return f"{sum(d['covered'] for d in days)} of {sum(d['total'] for d in days)}"


r3_rows = {}
for p in ("4", "6"):
    m, f = runs[p], full[p]
    r3_rows[f"Prompt {p}"] = {
        "Run-days without a summary": f"{m['failed_run_days']} of {m['run_days']}",
        "Pair score, all runs": f"{m['pair_total']} of {2 * m['pair_scores_count']}",
        "- shareholder acceptance and approval (23 October)": f"{m['pair_score']['2026-10-23']} of 6",
        "- results preview and actual results (24 October)": f"{m['pair_score']['2026-10-24']} of 6",
        "Industry points that extend the company's own facts to the sector": f"{m['overreach']} of {m['industry_points']}",
        "Events covered, all runs": covered(m),
        "Key points and briefs supported (one run per day)": f"{f['core']['supported']} of {f['core']['total']}",
        "Contradicted items (one run per day)": f["core"]["contradicted"],
        "Days with industry points despite no industry evidence (one run per day)": ", ".join(f["points_without_evidence_days"]) or "none",
    }
display(pd.DataFrame(r3_rows))
failed = [name for name, held in r3["choice"]["conditions"].items() if not held]
print("Conditions not met:", failed)
print("Consistency items:", {k: v for k, v in r3["decoys"].items() if k != "note"})
print("Prompt chosen:", r3["choice"]["prompt"])

,Prompt 4,Prompt 6
Run-days without a summary,0 of 9,0 of 9
"Pair score, all runs",8 of 12,9 of 12
- shareholder acceptance and approval (23 October),3 of 6,3 of 6
- results preview and actual results (24 October),5 of 6,6 of 6
Industry points that extend the company's own facts to the sector,2 of 7,0 of 4
"Events covered, all runs",31 of 33,33 of 33
Key points and briefs supported (one run per day),33 of 40,34 of 41
Contradicted items (one run per day),1,0
Days with industry points despite no industry evidence (one run per day),2026-10-14,2026-10-14


Conditions not met: ['no_industry_points_without_evidence']
Consistency items: {'decoys': 10, 'same_judgement': 8, 'same_contradicted': 10}
Prompt chosen: 4


**Prompt 4 stays.** Prompt 6 scores higher overall and is no worse on the other reported measures. It meets every choice condition but one. On 14 October, the industry evidence decision is "no", but prompt 6's run still writes an industry point about Europe's costly move to cleaner commercial vehicles. The rule requires every guardrail to hold, so prompt 4 stays and the freeze is unchanged.

This guardrail is not the overreach measure in the table. Overreach asks whether an industry point stretches the company's own facts to the sector. This guardrail asks whether a day whose articles hold no wider-industry evidence gets any industry point at all. Prompt 4's run for the same day has the same fault (last row of the table). The guardrail checks prompt 6 on its own instead of comparing it with prompt 4, so a weakness both prompts share decides the choice. The rule is fixed before the runs and is not changed after the result.

On the results pair (24 October), prompt 6 keeps the analysts' forecast and the actual results apart, with who expects the forecast, in all three runs; prompt 4 misses the source of the forecast once. On the acceptance pair (23 October), no summary of either prompt says who expects the outcome: every summary states the 80% acceptance as a requirement. Both prompts score 3 of 6 there.

The ten consistency items get the same judgement in 8 of 10 cases and the same contradiction decision in all 10.

In [25]:
import math
r3_rejudge = record("summary_v2_r3_rejudge_score.json")
levels = ["supported", "partly_supported", "not_supported"]
display(pd.DataFrame(r3_rejudge["table"]).T.loc[levels, levels].rename_axis("first \\ second"))

n = r3_rejudge["items"]
matches = sum(r3_rejudge["table"][level][level] for level in levels)
p, z = matches / n, 1.96
centre = (p + z * z / (2 * n)) / (1 + z * z / n)
half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
print(f"Matching support judgements: {matches} of {n} ({p:.1%}); approximate 95% interval {centre - half:.0%} to {centre + half:.0%}")
print(f"Matching contradiction decisions: {r3_rejudge['contradicted_agreement']:.0%}")

,supported,partly_supported,not_supported
first \ second,,,
supported,15,0,0
partly_supported,1,1,0
not_supported,0,0,0


Matching support judgements: 16 of 17 (94.1%); approximate 95% interval 73% to 99%
Matching contradiction decisions: 100%


A second blind judgement of 20% of the Round 3 items (17 items drawn with a fixed seed; decoys left out) gives the table above: 16 of 17 support judgements match the first (94.1%; approximate 95% interval 73% to 99%), and all 17 contradiction decisions match. The one difference is an industry point that turns one partnership into a wider trend ("increasingly being positioned as a hub"): partly supported in the first pass, supported in the second. This is the consistency of one judge across two passes, not agreement between independent judges, and it changes no judgement, score or choice.

## 11. Limitations

- **Free-plan limits.** The model runs on Groq's free plan: 8,000 tokens a minute and 200,000 a day. The minute limit rules out k = 10 with safe headroom (section 8), and the daily limit bounds how many runs and repeats are possible. NewsAPI's free plan shows articles about one day late, searches at most one month back and allows 100 requests a day, which limits the fresh data available for testing.
- **Small evaluation.** The design is chosen on three development runs and tested on one fresh week, with one summary each. The samples are small (15 items for the second judgement, 14 items and 4 events on the fresh week), so every share in this notebook carries wide uncertainty.
- **One judge.** Every judgement comes from one judge. A second pass by the same judge matches on 13 of 15 items in the design comparison and on 16 of 17 items in the third prompt test; an independent second judge is not available in this project.
- **Industry-context trade-off.** On the development runs, the stricter rule (prompt version 4) misses a relevant fact about another company in the same sector (one case in three runs), and the looser rule (prompt version 5) adds weak points. Prompt version 4 is selected; the fresh week shows it reporting real sector facts without unsupported claims, on one summary only.
- **One shared fault decides the prompt choice.** In the third test (10.3), prompt 6 scores higher overall, but an absolute guardrail blocks it for an industry point that prompt 4 also writes. A relative guardrail could treat a shared fault differently, but that would be a new choice rule and would need a separately planned test.
- **One sample per run.** The model writes with temperature 1.0, so the same articles can give a different summary on another run. Every result in this notebook is one sample, not an average over repeats.
- **What the automatic checks cannot see.** The number check confirms that a number appears in the cited article, not that it belongs to the same company, measure, period and direction (section 2). Those errors are found only by the human judging.
- **Publisher text stays outside Git.** Article texts and the judging pages are kept in `data/fulltext/`, which Git ignores. The committed records, sheets and fingerprints preserve the results and allow the integrity of the saved evidence to be checked. Independently checking article support or rebuilding a judging page requires the article texts on the computer that runs it.